# Red B: encontrarla, reevaluarla y revisar el sistema final

La reevaluación con 300 semillas no encontró la red B (imitación + PPO, semilla 2000), aunque el archivo existe. Este cuaderno:

1. **Busca** en Drive el checkpoint de la red B (semilla 2000, 1.000 episodios, tasa 0,0001) y lo deja en la carpeta que espera la reevaluación. Si no lo encuentra, te pide **subir** el `checkpoint_last.pt` desde el PC.
2. **Reevalúa** solo esa red en las 300 semillas (≈ 12 min). Lo demás ya está hecho y se salta.
3. **Vuelve a aplicar la regla** del Ejecutor. Si sigue ganando la red A, no cambia nada. Si gana la B, guarda el Maestro anterior aparte y lo **vuelve a entrenar** con la red B (≈ 15 min).

**Cómo usarlo:** «Ejecutar todas». Al final arma `reeval_red_B.zip`.

## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama feature/ps-cierre-sprint6; debe mostrar un commit y un número mayor que 0)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && git checkout -q -f origin/feature/ps-cierre-sprint6
!cd /content/slippage && git log --oneline -1 && grep -c "imitar_heuristica" scripts/colab/entrenamiento_inicial_ps.py


In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## 1. Encontrar la red B

In [ ]:
# Buscar el checkpoint de la red B y dejarlo donde lo busca la reevaluación
import glob, os, shutil, torch
destino = f'{RESULTS_DIR}/rimit/d3_opcion0_s2000/checkpoints_apertura/checkpoint_last.pt'

def leer(p):
    try:
        return torch.load(p, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(p, map_location='cpu')

def es_red_b(c):
    return (c.get('seed') == 2000 and c.get('current_episode') == 1000 and c.get('role') == 'ejecutor'
            and abs(float(c.get('cfg', {}).get('lr', 0)) - 1e-4) < 1e-9)

candidatos = []
for p in glob.glob(f'{RESULTS_DIR}/**/*.pt', recursive=True):
    if 'rimit_bc' in p or 'sistema_final' in p:
        continue
    try:
        c = leer(p)
    except Exception as e:
        continue
    if isinstance(c, dict) and es_red_b(c):
        candidatos.append(p)
print('Checkpoints de la red B encontrados:', candidatos or 'ninguno')

if os.path.exists(destino) and es_red_b(leer(destino)):
    print('OK: la red B ya está en', destino)
elif candidatos:
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    shutil.copy(candidatos[0], destino)
    print('Copiado', candidatos[0], '->', destino)
else:
    print('No está en Drive. Sube el checkpoint_last.pt de la red B (el que tienes en el PC):')
    from google.colab import files
    subido = files.upload()
    nombre = next(iter(subido))
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    shutil.move(nombre, destino)
    assert es_red_b(leer(destino)), 'El archivo subido no es la red B (semilla 2000, 1.000 episodios)'
    print('Subido a', destino)


## 2. Reevaluarla (≈ 12 min)

In [ ]:
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/reevaluar_ejecutores_ps.py --raices {RESULTS_DIR}/rimit --n-semillas 300 --out-dir {RESULTS_DIR}/reeval_300 --max-minutes 200 2>&1 | grep -v UserWarning
import os
assert os.path.exists(f'{RESULTS_DIR}/reeval_300/red_rimit__d3_opcion0_s2000.json'), 'La red B no quedó evaluada: mándale la pantalla a PS'


## 3. Regla del Ejecutor y, si cambia, Maestro de nuevo

In [ ]:
# Aplicar la regla otra vez y, si cambia el Ejecutor, rehacer el Maestro
import json, os, shutil
sf = f'{RESULTS_DIR}/sistema_final'
antes = json.load(open(f'{sf}/ejecutor_elegido.json', encoding='utf-8'))['elegida']['etiqueta']
revision = f'{RESULTS_DIR}/sistema_final_revision'
get_ipython().system(f"cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/elegir_ejecutor_ps.py --reeval-dir {RESULTS_DIR}/reeval_300 --results-dir {RESULTS_DIR} --out {revision}")
ahora = json.load(open(f'{revision}/ejecutor_elegido.json', encoding='utf-8'))['elegida']
print('Antes:', antes, '| Ahora:', ahora['etiqueta'], f"({ahora['slippage_bps']:.2f} bps)")
if ahora['etiqueta'] == antes:
    print('SIN CAMBIOS: el sistema final sigue igual. Puedes correr las 30 corridas.')
else:
    print('CAMBIA el Ejecutor: se guarda el Maestro anterior y se vuelve a entrenar (≈ 15 min).')
    viejo = f'{sf}/maestro_con_{antes}'
    if os.path.exists(f'{sf}/maestro') and not os.path.exists(viejo):
        shutil.move(f'{sf}/maestro', viejo)
    os.makedirs(viejo, exist_ok=True)
    shutil.copy(f'{sf}/ejecutor_elegido.json', f'{viejo}/ejecutor_elegido_anterior.json')
    get_ipython().system(f"cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/elegir_ejecutor_ps.py --reeval-dir {RESULTS_DIR}/reeval_300 --results-dir {RESULTS_DIR} --out {sf}")
    get_ipython().system(f"cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/entrenar_maestro.py --jornadas 2000 --impacto --inicio-aleatorio --lambda-penalty 0.02 --q-total 88300 --volumen-dia 882768 --ckpt-ejecutor {sf}/ejecutor_elegido.pt --out-dir {sf}/maestro 2>&1 | grep -v UserWarning | tail -6")
    get_ipython().system(f"cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/revalidar_lambda_maestro.py --resumen {sf}/maestro/resumen_maestro.json --out {sf}/maestro/lambda.json")
    print('Listo: sistema final rehecho con', ahora['etiqueta'], '. Ahora sí, las 30 corridas.')


## Zip para descargar

In [ ]:
import os, zipfile
archivos = [f'{RESULTS_DIR}/reeval_300/red_rimit__d3_opcion0_s2000.json']
for base in (f'{RESULTS_DIR}/sistema_final', f'{RESULTS_DIR}/sistema_final_revision'):
    archivos += [os.path.join(c, a) for c, _, fs in os.walk(base) for a in fs if a.endswith(('.json', '.csv'))]
zpath = f'{RESULTS_DIR}/reeval_red_B.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a, os.path.relpath(a, RESULTS_DIR))
print(len(archivos), 'archivos -> zip listo:', zpath)
